# AI vs Real Image Classification — CLIP-L/14 Backbone

Dataset: `./data/raw/RDDataset_final`

```
RDDataset_final
├── original
│   ├── ai
│   └── real
├── redigital
│   ├── ai
│   └── real
└── transfer
    ├── ai
    └── real
```

The three top-level folders (`original`, `redigital`, `transfer`) describe how the image was
produced/processed and are kept only as metadata. The classification **target** is the leaf
folder name: `ai` vs `real`.

Steps covered in this notebook:
1. Build an index of every image in the dataset.
2. Draw a class-balanced subset (equal number of `ai` / `real` images).
3. Instantiate a pretrained CLIP ViT-L/14 ("CLIP/L") backbone with a classification head for `ai` vs `real`.
4. Split into train / validation / test, train the head, and evaluate on the held-out test set.

> Requires: `torch`, `torchvision`, `transformers`, `pandas`, `numpy`, `Pillow`, `scikit-learn`, `matplotlib`, `tqdm`.
> If needed: `pip install torch torchvision transformers pandas numpy pillow scikit-learn matplotlib tqdm`


## Imports

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    CLIPModel, CLIPImageProcessor,
    SwinModel, AutoImageProcessor,
    DeiTModel, DeiTImageProcessor,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
)

from tqdm.auto import tqdm
import copy

## Globals

In [2]:
# --- Dataset structure -------------------------------------------------
DATA_ROOT = Path("./data/raw/RRDataset_final")
SUBFOLDERS = ["original", "redigital", "transfer"]   # processing type, kept as metadata only
CLASS_NAMES = ["ai", "real"]                          # classification target
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# --- Sampling ------------------------------------------------------------
SAMPLES_PER_CLASS = 750   # images per class (ai / real) to load; set to None to use ALL images
SEED = 42

# --- Model -----------------------------------------------------------------
MODEL_CONFIGS = {
    "clip-vit-l": {
        "type":       "clip",
        "name":       "openai/clip-vit-large-patch14",
    },
    "clip-vit-b": {
        "type":       "clip",
        "name":       "openai/clip-vit-base-patch32",
    },
    "swin-tiny": {
        "type":       "swin",
        "name":       "microsoft/swin-tiny-patch4-window7-224",
    },
    "deit-tiny": {
        "type":       "deit",
        "name":       "facebook/deit-tiny-patch16-224",
    },
    "deit-small": {
        "type":       "deit",
        "name":       "facebook/deit-small-patch16-224",
    },
}

NUM_CLASSES = len(CLASS_NAMES)
CHOSEN_MODEL = "deit-small"
CLIP_MODEL_NAME = MODEL_CONFIGS[CHOSEN_MODEL]["name"]
FREEZE_BACKBONE = True   # freeze CLIP weights, train only the classification head

# --- Hardware ----------------------------------------------------------
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# --- Dataloader ----------------------------------------------------------
BATCH_SIZE = 32
NUM_WORKERS = 4

# --- Train / validation / test split -------------------------------------
TEST_SIZE = 0.2     # fraction of the balanced subset held out as the final test set
VAL_SIZE = 0.1       # fraction of the remaining train+val data used for validation during training

# --- Training --------------------------------------------------------------
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_PATH = CHECKPOINT_DIR / "clip_ai_real_head_best.pt"

print(f"Device: {DEVICE}")


Device: cuda


## Utils

In [3]:
def set_seed(seed: int = SEED) -> None:
    """Make sampling / shuffling reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


## Data

### 1. Build a full index of the dataset

Walk every `subfolder/class/*` directory and record the path, the target label
(`ai` / `real`) and the originating subfolder (kept for inspection only).

In [4]:
def build_image_index(data_root: Path = DATA_ROOT) -> pd.DataFrame:
    """
    Walk the dataset folder structure:

        data_root / <subfolder> / <class> / *.jpg

    and return a DataFrame with one row per image:
        path        - absolute path to the image file
        label       - "ai" or "real" (the classification target)
        subfolder   - "original" / "redigital" / "transfer" (metadata only)
    """
    records = []
    for subfolder in SUBFOLDERS:
        for class_name in CLASS_NAMES:
            class_dir = data_root / subfolder / class_name
            if not class_dir.is_dir():
                print(f"[warning] missing folder: {class_dir}")
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.is_file() and is_image_file(img_path):
                    records.append(
                        {
                            "path": str(img_path),
                            "label": class_name,
                            "subfolder": subfolder,
                        }
                    )

    df = pd.DataFrame(records)
    if df.empty:
        raise RuntimeError(f"No images found under {data_root}. Check the dataset path/structure.")
    return df


full_index = build_image_index()
print(f"Total images found: {len(full_index)}")
print(full_index.groupby("label").size())
print(full_index.groupby(["subfolder", "label"]).size())


Total images found: 50999
label
ai      25500
real    25499
dtype: int64
subfolder  label
original   ai       8500
           real     8500
redigital  ai       8500
           real     8499
transfer   ai       8500
           real     8500
dtype: int64


### 2. Draw a class-balanced subset

Only a portion of the full dataset is loaded (`SAMPLES_PER_CLASS` images per class).
Within each class, the sample is drawn proportionally across the three subfolders so the
subset stays representative of the original data, while keeping an exact 50/50 `ai`/`real`
split.

In [5]:
def sample_balanced_subset(
    df: pd.DataFrame,
    samples_per_class: "int | None" = SAMPLES_PER_CLASS,
    seed: int = SEED,
) -> pd.DataFrame:
    """Take an equally-sized, class-balanced subset of the full index."""
    subsets = []

    for class_name in CLASS_NAMES:
        class_df = df[df["label"] == class_name]
        n_available = len(class_df)
        n_take = n_available if samples_per_class is None else min(samples_per_class, n_available)

        sampled = class_df.sample(n=n_take, random_state=seed)
        subsets.append(sampled)

        if samples_per_class is not None and n_available < samples_per_class:
            print(f"[warning] class '{class_name}' has only {n_available} images, "
                  f"fewer than the requested {samples_per_class}.")

    subset_df = pd.concat(subsets, ignore_index=True)
    subset_df = subset_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)  # shuffle
    return subset_df


set_seed(SEED)
data_subset = sample_balanced_subset(full_index, SAMPLES_PER_CLASS, SEED)
print(f"Subset size: {len(data_subset)}")
print(data_subset.groupby("label").size())
print(data_subset.groupby(["subfolder", "label"]).size())


Subset size: 1500
label
ai      750
real    750
dtype: int64
subfolder  label
original   ai       254
           real     241
redigital  ai       237
           real     255
transfer   ai       259
           real     254
dtype: int64


### 3. Train / validation / test split

The balanced subset is split into three disjoint, stratified parts:
- **test** (`TEST_SIZE`) — held out untouched until the `Evaluation` section.
- **train** / **val** — the remainder, split again (`VAL_SIZE`) so progress can be monitored
  after every training epoch without touching the test set.

In [6]:
train_val_df, test_df = train_test_split(
    data_subset,
    test_size=TEST_SIZE,
    stratify=data_subset["label"],
    random_state=SEED,
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=VAL_SIZE,
    stratify=train_val_df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name}:", dict(split_df["label"].value_counts()))


Train: 1080  |  Val: 120  |  Test: 300
  train: {'real': np.int64(540), 'ai': np.int64(540)}
  val: {'real': np.int64(60), 'ai': np.int64(60)}
  test: {'ai': np.int64(150), 'real': np.int64(150)}


### 4. PyTorch `Dataset` / `DataLoader`

Images are preprocessed with CLIP's own image processor (resize, center-crop, normalization
matching the pretrained weights).

In [7]:
class AIRealDataset(Dataset):
    """Loads images and returns CLIP-preprocessed pixel tensors + integer labels."""

    def __init__(self, dataframe: pd.DataFrame, image_processor: CLIPImageProcessor):
        self.df = dataframe.reset_index(drop=True)
        self.image_processor = image_processor

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")

        pixel_values = self.image_processor(images=image, return_tensors="pt")["pixel_values"][0]
        label = CLASS_TO_IDX[row["label"]]

        return pixel_values, label


clip_image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_NAME)

train_dataset = AIRealDataset(train_df, clip_image_processor)
val_dataset = AIRealDataset(val_df, clip_image_processor)
test_dataset = AIRealDataset(test_df, clip_image_processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# quick sanity check
sample_pixels, sample_label = train_dataset[0]
print(f"Sample tensor shape: {sample_pixels.shape}, label: {sample_label} ({CLASS_NAMES[sample_label]})")


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Sample tensor shape: torch.Size([3, 224, 224]), label: 1 (real)


## Network

Pretrained CLIP ViT-L/14 vision encoder (`openai/clip-vit-large-patch14`) + a small MLP
classification head producing `ai` / `real` logits. The backbone is frozen by default
(`FREEZE_BACKBONE = True`); set it to `False` to fine-tune the whole encoder.

In [8]:
class TransformerClassifier(nn.Module):
    """
    Unified classifier supporting CLIP ViT-B, Swin-Tiny, DeiT-Tiny/Small.
    backbone_type: 'clip' | 'swin' | 'deit'
    """

    def __init__(self, cfg: dict, num_classes: int = NUM_CLASSES,
                 freeze_backbone: bool = FREEZE_BACKBONE):
        super().__init__()
        btype = cfg["type"]
        hf_name = cfg["name"]

        if btype == "clip":
            _clip = CLIPModel.from_pretrained(hf_name)
            self.backbone        = _clip.vision_model
            self.projection      = _clip.visual_projection
            embed_dim            = _clip.config.projection_dim          # 512 for ViT-B/32
            self._forward_fn     = self._clip_forward

        elif btype == "swin":
            self.backbone        = SwinModel.from_pretrained(hf_name)
            self.projection      = None
            embed_dim            = self.backbone.config.hidden_size     # 768 for Swin-Tiny
            self._forward_fn     = self._swin_forward

        elif btype == "deit":
            self.backbone        = DeiTModel.from_pretrained(
                                       hf_name, add_pooling_layer=True)
            self.projection      = None
            embed_dim            = self.backbone.config.hidden_size     # 192/384
            self._forward_fn     = self._deit_forward

        else:
            raise ValueError(f"Unknown backbone type: {btype!r}")

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
            if self.projection is not None:
                for p in self.projection.parameters():
                    p.requires_grad = False

        self.head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    # --- per-backbone pooling helpers ---
    def _clip_forward(self, x):
        out = self.backbone(pixel_values=x).pooler_output
        return self.projection(out)                     # [B, embed_dim]

    def _swin_forward(self, x):
        return self.backbone(pixel_values=x).pooler_output   # [B, embed_dim]

    def _deit_forward(self, x):
        return self.backbone(pixel_values=x).pooler_output   # [B, embed_dim]

    def forward(self, x):
        features = self._forward_fn(x)
        return self.head(features)


model = TransformerClassifier(MODEL_CONFIGS[CHOSEN_MODEL]).to(DEVICE)

n_total_params = sum(p.numel() for p in model.parameters())
n_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {n_total_params:,}")
print(f"Trainable parameters: {n_trainable_params:,}  (backbone frozen: {FREEZE_BACKBONE})")


config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

[transformers] You are using a model of type `vit` to instantiate a model of type `deit`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


pytorch_model.bin:   0%|          | 0.00/88.3M [00:00<?, ?B/s]

Loading weights: 0it [00:00, ?it/s]

[transformers] DeiTModel LOAD REPORT from: facebook/deit-small-patch16-224
Key                                               | Status     | 
--------------------------------------------------+------------+-
vit.layers.{0...11}.layernorm_after.bias          | UNEXPECTED | 
vit.layers.{0...11}.layernorm_after.weight        | UNEXPECTED | 
vit.layers.{0...11}.mlp.fc2.weight                | UNEXPECTED | 
vit.layers.{0...11}.attention.o_proj.bias         | UNEXPECTED | 
vit.layers.{0...11}.attention.v_proj.bias         | UNEXPECTED | 
vit.layers.{0...11}.mlp.fc1.weight                | UNEXPECTED | 
vit.layers.{0...11}.attention.v_proj.weight       | UNEXPECTED | 
vit.layers.{0...11}.mlp.fc2.bias                  | UNEXPECTED | 
vit.layers.{0...11}.mlp.fc1.bias                  | UNEXPECTED | 
vit.layers.{0...11}.layernorm_before.weight       | UNEXPECTED | 
vit.layers.{0...11}.attention.o_proj.weight       | UNEXPECTED | 
vit.layers.{0...11}.attention.k_proj.bias         | UNEXPECTED | 
v

Total parameters:     21,913,346
Trainable parameters: 99,074  (backbone frozen: True)


### Sanity check — forward pass on one batch

In [9]:
model.eval()
with torch.no_grad():
    pixel_values, labels = next(iter(train_loader))
    pixel_values = pixel_values.to(DEVICE)
    logits = model(pixel_values)
    print(f"Logits shape: {logits.shape}")  # [BATCH_SIZE, NUM_CLASSES]


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Logits shape: torch.Size([32, 2])


## Train

Only the classification head has `requires_grad=True` while `FREEZE_BACKBONE = True`
(set it to `False` in **Globals** to fine-tune the whole CLIP encoder too — lower the
learning rate if you do).

### 1. Loss, optimizer, epoch runner

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


def run_epoch(model, loader, criterion, optimizer=None):
    """
    Run one pass over `loader`.

    If `optimizer` is given, the model is set to train mode and weights are updated
    (a training epoch). Otherwise the model is set to eval mode and no gradients are
    computed (a validation / test epoch).

    Returns: (average_loss, accuracy)
    """
    is_training = optimizer is not None
    model.train(is_training)

    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_training):
        for pixel_values, labels in tqdm(loader, leave=False):
            pixel_values = pixel_values.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(pixel_values)
            loss = criterion(logits, labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


### 2. Training loop

Trains for `NUM_EPOCHS`, evaluating on the validation split after every epoch. The
best-performing weights (by validation accuracy) are checkpointed to `CHECKPOINT_PATH`
and reloaded into `model` at the end.

In [ ]:
set_seed(SEED)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_state_dict = None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())
        torch.save(best_state_dict, CHECKPOINT_PATH)
        print(f"  -> new best val_acc={best_val_acc:.4f}, checkpoint saved to {CHECKPOINT_PATH}")

# reload the best-performing weights before evaluation
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
print(f"\nBest validation accuracy: {best_val_acc:.4f}")


  0%|          | 0/34 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

Epoch 01/10 | train_loss=0.6864 train_acc=0.5333 | val_loss=0.6794 val_acc=0.6083
  -> new best val_acc=0.6083, checkpoint saved to checkpoints/clip_ai_real_head_best.pt


  0%|          | 0/34 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...


AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78d967fbad40>
Traceback (most recent call last):
      File "/home/tiziano/venv/cv/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1686, in __del__
self._shutdown_workers()
  File "/home/tiziano/venv/cv/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1669, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78d967fbad40>
Traceback (most recent call last):
  File "/home/tiziano/venv/cv/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1686, in __del__
    self._shutdown_workers()
  File "/home/tiziano/venv/cv/lib/python3.13/site-packages/torch/utils/

  0%|          | 0/4 [00:00<?, ?it/s]

Epoch 06/10 | train_loss=0.6271 train_acc=0.6620 | val_loss=0.6615 val_acc=0.5917


  0%|          | 0/34 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78d967fbad40>
Traceback (most recent call last):
  File "/home/tiziano/venv/cv/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1686, in __del__
    self._shutdown_workers()
  File "/home/tiziano/venv/cv/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1669, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
Exception ignored in:     assert self._parent_pid == os.getpid(), 'can only test a child process'<function _MultiProcessingDataLoaderIter.__del__ at 0x78d967fbad40>

AssertionErrorTraceback (most recent call last):
:   File "/home/tiziano/venv/cv/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1686, in __del__
can only test a child process    self._shutdown_workers()

  File "/home/tiziano/venv/cv/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1669, in _shutdown_wo

### 3. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## Evaluation

### 1. Predictions on the held-out test set

In [ ]:
def predict(model, loader):
    """Run inference and return (true_labels, predicted_labels, prob_of_'real')."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for pixel_values, labels in tqdm(loader, leave=False):
            pixel_values = pixel_values.to(DEVICE)
            logits = model(pixel_values)
            probs = torch.softmax(logits, dim=1)

            all_labels.extend(labels.numpy())
            all_preds.extend(probs.argmax(dim=1).cpu().numpy())
            all_probs.extend(probs[:, CLASS_TO_IDX["real"]].cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


y_true, y_pred, y_prob_real = predict(model, test_loader)
test_acc = accuracy_score(y_true, y_pred)
print(f"Test accuracy: {test_acc:.4f}")


### 2. Classification report & confusion matrix

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix — test set")
plt.show()


### 3. ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob_real, pos_label=CLASS_TO_IDX["real"])
auc = roc_auc_score(y_true, y_prob_real)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve — real vs ai")
plt.legend()
plt.show()


### 4. A few misclassified examples

In [ ]:
misclassified_idx = np.where(y_true != y_pred)[0]
n_show = min(8, len(misclassified_idx))

if n_show == 0:
    print("No misclassified examples in the test set!")
else:
    show_idx = np.random.RandomState(SEED).choice(misclassified_idx, size=n_show, replace=False)

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, idx in zip(axes.flat, show_idx):
        row = test_df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        ax.imshow(image)
        ax.set_title(f"true: {CLASS_NAMES[y_true[idx]]} | pred: {CLASS_NAMES[y_pred[idx]]}", fontsize=9)
        ax.axis("off")
    for ax in axes.flat[n_show:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


---
The best-validation-accuracy weights are saved at `CHECKPOINT_PATH`. To reuse them in a new
session without retraining: re-run **Imports** → **Globals** → **Network**, then call
`model.load_state_dict(torch.load(CHECKPOINT_PATH))`.

In [ ]:
# ----------------------------------------------------------------------
# Misclassification vs. transformation type (original / redigital / transfer)
# ----------------------------------------------------------------------
from scipy.stats import chi2_contingency

results_df = test_df.copy()
results_df["true_label"] = [CLASS_NAMES[i] for i in y_true]
results_df["pred_label"] = [CLASS_NAMES[i] for i in y_pred]
results_df["correct"] = results_df["true_label"] == results_df["pred_label"]

# error rate per transformation type x true class
error_by_subfolder = (
    results_df.groupby(["subfolder", "true_label"])["correct"]
    .agg(n="size", accuracy="mean")
    .reset_index()
)
error_by_subfolder["error_rate"] = 1 - error_by_subfolder["accuracy"]
print(error_by_subfolder)

# grouped bar chart: error rate by transformation type, split by true class
pivot = error_by_subfolder.pivot(index="subfolder", columns="true_label", values="error_rate")
pivot = pivot.reindex(SUBFOLDERS)

ax = pivot.plot(kind="bar", figsize=(7, 4), rot=0)
ax.set_ylabel("Misclassification rate")
ax.set_xlabel("Transformation (subfolder)")
ax.set_title("Misclassification rate by transformation type and true class")
ax.legend(title="true label")
plt.tight_layout()
plt.show()

# is misclassification independent of the transformation type, or correlated with it?
contingency = pd.crosstab(results_df["subfolder"], results_df["correct"])
print("\nContingency table (subfolder vs. correct):")
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)
n = contingency.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print(f"\nchi2 = {chi2:.3f}, dof = {dof}, p-value = {p_value:.4f}")
print(f"Cramér's V = {cramers_v:.3f}  (0 = no association, 1 = perfect association)")
if p_value < 0.05:
    print("-> Misclassification rate depends on the transformation type (p < 0.05).")
else:
    print("-> No statistically significant association found at the 0.05 level.")